# Manuscript figure panels

Panels in this notebook follow their order and labels in the manuscript. Add later code-generated panels below the existing ones in manuscript order. The older `paper_figure_panels.ipynb` uses internal panel numbers and remains a source of analysis recipes; its numbers do not identify manuscript panels.

## Index

- [Figure 1e — training reward preference index](#figure-1e)
- [Figure 1f — post-period reward preference index](#figure-1f)
- [Figure 1g — training spatial learning index](#figure-1g)
- [Figure 1h — post-period spatial learning index](#figure-1h)
- [Figure 1i — reward rate versus spatial learning index](#figure-1i)
- [Copy manuscript panels](#copy-manuscript-panels)


## Setup

Run the setup and plot-style cells first. Analysis is opt-in: the panel cell previews the command until `RUN_FIGURE_1E_1H` is set to `True`. The input videos must be available at the Synology paths in the command. Run the notebook with the project's analysis environment, which supplies `analyze.py` dependencies.


## Plot style

Set the font family and image format here. Time plots default to 27 pt; correlation plots default to 20 pt. Each recipe declares its plot type, so the matching font size is applied when its command runs. Rerun this cell and then the desired panel cell after changing a setting.


In [ ]:
PLOT_STYLE = {
    'font_family': 'Arial',
    'font_size_time': 27,
    'font_size_correlation': 20,
    'image_format': 'pdf',
}
PLOT_STYLE


In [ ]:
from pathlib import Path
import os
import re
import shlex
import shutil
import subprocess

from IPython.display import Image, Markdown, display

ROOT = Path.cwd().resolve()
if not (ROOT / 'analyze.py').is_file():
    ROOT = ROOT.parent
if not (ROOT / 'analyze.py').is_file():
    raise FileNotFoundError('Start Jupyter in the repository root or notebooks directory.')
os.chdir(ROOT)
display(Markdown(f'Working directory: `{ROOT}`'))

def styled_command(recipe):
    font_size = PLOT_STYLE['font_size_' + recipe['plot_type']]
    image_format = PLOT_STYLE['image_format'].strip().lstrip('.').lower()
    if not image_format:
        raise ValueError('image_format must not be empty')
    tokens = shlex.split(recipe['command'])
    tokens += ['--fs', str(font_size), '--fontFamily', PLOT_STYLE['font_family'],
               '--imgFormat', image_format]
    return shlex.join(tokens), image_format

def run_manuscript_recipe(recipe, run=False):
    command, image_format = styled_command(recipe)
    display(Markdown(f'### {recipe["title"]}'))
    display(Markdown('```bash\n' + command + '\n```'))
    for panel, paths in recipe['panels'].items():
        source = Path(paths['source']).with_suffix('.' + image_format)
        output = Path(paths['output']).with_suffix('.' + image_format)
        display(Markdown(f'**Figure {panel}:** `{source}` → `{output}`'))
    if not run:
        display(Markdown('_Preview only. Set the run toggle to `True` to run this analysis._'))
        return
    subprocess.run(command, shell=True, cwd=ROOT, check=True)
    for panel, paths in recipe['panels'].items():
        source = ROOT / Path(paths['source']).with_suffix('.' + image_format)
        output = ROOT / Path(paths['output']).with_suffix('.' + image_format)
        if not source.is_file():
            raise FileNotFoundError(f'Figure {panel} source plot was not generated: {source}')
        output.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, output)
        display(Markdown(f'Figure {panel} saved: `{output}`'))
        if output.suffix.lower() in {'.png', '.jpg', '.jpeg', '.gif'}:
            display(Image(filename=str(output)))


## Figure 1e

**Reward preference index during training 1 and 2.** Control flies (UAS-CsChrimson; 0273-Gal4) in the flat high-throughput learning (HTL) chamber. The analysis command uses flies 0–9 and a control-circle radius multiplier of 5. `analyze.py` produces the training plot in 10-minute buckets.

The video selection matches the flat, sighted control cohort in `video_lists.log`. The same analysis run produces Figures 1f–1h.


## Figure 1f

**Reward preference index during post-periods 1 and 2.** The same flies and analysis command as Figure 1e. `analyze.py` produces the post-period plot in 3-minute buckets. Figures 1e and 1f show reward PI (`reward_pi`); Figures 1g and 1h show the experimental-minus-yoked difference (`reward_pi_diff`).


## Figure 1g

**Spatial learning index during training 1 and 2.** For the same flat HTL control flies, this is experimental reward PI minus yoked reward PI, from `reward_pi_diff__10_min_buckets.pdf`. It uses the same 10-minute training buckets and analysis command as Figure 1e.


## Figure 1h

**Spatial learning index during post-periods 1 and 2.** This is the corresponding experimental-minus-yoked reward PI difference from `reward_pi_post_diff__3_min_buckets.pdf`, using 3-minute post-period buckets.


In [ ]:
figure_1e_1h = {
    'title': 'Figures 1e–1h — flat HTL control reward PI and SLI',
    'command': (
        "python analyze.py -v '/media/Synology4/Yang Chen/2024-03-04/c3[12]_*,"
        "/media/Synology4/Yang Chen/2024-03-04/c4[12]_*,"
        "/media/Synology4/Yang Chen/2024-03-14/c4[12]_*,"
        "/media/Synology4/Yang Chen/2024-03-18/c5[12]_*,"
        "/media/Synology4/Yang Chen/2024-03-18/c6[12]_*,"
        "/media/Synology4/Yang Chen/2024-06-12/c[12]_*' -f 0-9 --rmCC 5 --num-trainings 2"
    ),
    'plot_type': 'time',
    'panels': {
        '1e': {
            'source': 'imgs/reward_pi__10_min_buckets.png',
            'output': 'imgs/manuscript/fig1e_reward_pi_training_flat_htl_control.png',
        },
        '1f': {
            'source': 'imgs/reward_pi_post__3_min_buckets.png',
            'output': 'imgs/manuscript/fig1f_reward_pi_post_flat_htl_control.png',
        },
        '1g': {
            'source': 'imgs/reward_pi_diff__10_min_buckets.png',
            'output': 'imgs/manuscript/fig1g_sli_training_flat_htl_control.png',
        },
        '1h': {
            'source': 'imgs/reward_pi_post_diff__3_min_buckets.png',
            'output': 'imgs/manuscript/fig1h_sli_post_flat_htl_control.png',
        },
    },
}

RUN_FIGURE_1E_1H = False
run_manuscript_recipe(figure_1e_1h, run=RUN_FIGURE_1E_1H)


## Figure 1i

**Reward rate versus spatial learning index (SLI).** This uses the same flat HTL control flies and base analysis options as Figures 1e–1h. SLI is averaged over **training 2, sync buckets 2–5** by selecting training 2, using the training mean, and skipping the first sync bucket. Reward rate inherits that same window. The selected plot is `corr_rpt_vs_sli_sliT2_mean_skip1__rptT2_mean_skip1.pdf`; the plot-style cell applies the 20 pt correlation font size.


In [ ]:
figure_1i = {
    'title': 'Figure 1i — reward rate versus mean SLI, T2 sync buckets 2–5',
    'command': (
        figure_1e_1h['command']
        + ' --sli-use-training-mean --best-worst-trn 2'
          ' --sli-select-skip-first-sync-buckets 1'
    ),
    'plot_type': 'correlation',
    'panels': {
        '1i': {
            'source': 'imgs/correlations/corr_rpt_vs_sli_sliT2_mean_skip1__rptT2_mean_skip1.png',
            'output': 'imgs/manuscript/fig1i_reward_rate_vs_sli_flat_htl_control.png',
        },
    },
}

RUN_FIGURE_1I = False
run_manuscript_recipe(figure_1i, run=RUN_FIGURE_1I)


## Copy manuscript panels

Copy generated panel files to the manuscript layout on Synology. The destination uses the manuscript figure number and panel label, for example `fig1/1e.pdf` through `fig1/1h.pdf`. This stage previews the mapping by default. Add each later recipe to `MANUSCRIPT_RECIPES` after defining it above. The destination directory must already be mounted.


In [ ]:
DEFAULT_MANUSCRIPT_COPY_DIR = Path('/media/Synology4/Robert/nvsl-analysis-plots')

def copy_manuscript_panels(recipes, target_dir=DEFAULT_MANUSCRIPT_COPY_DIR, run=False):
    target_dir = Path(target_dir)
    copies = []
    seen = set()
    for recipe in recipes:
        _, image_format = styled_command(recipe)
        for panel, paths in recipe['panels'].items():
            match = re.fullmatch(r'(\d+)[a-z]?', panel)
            if not match:
                raise ValueError(f'Invalid manuscript panel label: {panel}')
            if panel in seen:
                raise ValueError(f'Duplicate manuscript panel label: {panel}')
            seen.add(panel)
            source = ROOT / Path(paths['output']).with_suffix('.' + image_format)
            destination = target_dir / f'fig{match.group(1)}' / f'{panel}.{image_format}'
            copies.append((source, destination))

    display(Markdown('### Copy destinations'))
    for source, destination in copies:
        display(Markdown(f'`{source}` → `{destination}`'))
    if not run:
        display(Markdown('_Preview only. Set `COPY_MANUSCRIPT_PANELS = True` to copy._'))
        return
    if not target_dir.is_dir():
        display(Markdown(f'_Copy skipped: destination mount unavailable: `{target_dir}`_'))
        return
    missing = [source for source, _ in copies if not source.is_file()]
    if missing:
        raise FileNotFoundError('Missing generated panel files: ' + ', '.join(map(str, missing)))
    for source, destination in copies:
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, destination)
        display(Markdown(f'Copied: `{destination}`'))

MANUSCRIPT_RECIPES = [figure_1e_1h, figure_1i]
MANUSCRIPT_COPY_DIR = DEFAULT_MANUSCRIPT_COPY_DIR
COPY_MANUSCRIPT_PANELS = False
copy_manuscript_panels(MANUSCRIPT_RECIPES, MANUSCRIPT_COPY_DIR, run=COPY_MANUSCRIPT_PANELS)
